In [1]:
import warnings

import numpy as np
import pandas as pd
import janitor
import requests
import pyreadr

warnings.filterwarnings("ignore", category=FutureWarning)

FES_CSV = "../data/country_fes.csv"
WB_API_ENDPOINT = "https://api.worldbank.org/v2/country/{}/indicator/NY.GDP.PCAP.CD"
# UN EGDI, https://www.qogdata.pol.gu.se/data/qog_std_ts_jan26.csv
QOG_CSV = "../data/qog_extract.csv"
# V-Dem electoral democracy index, https://ourworldindata.org/grapher/electoral-democracy-index
VDEM_URL = "https://ourworldindata.org/grapher/electoral-democracy-index.csv"
# GCI 2020, https://statbase.org
GCI_CSV = "../data/gci_2020.csv"
# CEPII GeoDist, via the cepiigeodist R package
CEPII_CSV = "../data/cepii_geo.csv"

OUT_CSV = "../data/country_fes_covariates.csv"

df_fes = pd.read_csv(FES_CSV)
df_fes.head(3)

,cc3,country,fe_breach_serious,fe_breach
0,AUS,Australia,0.891,0.941
1,GBR,UK,0.798,0.830
2,JEY,Jersey,0.788,0.879


## WB - GDPPC

In [2]:
# JEY/GGY are not in World Bank data
wb_codes = ";".join(c for c in df_fes["cc3"] if c not in {"JEY", "GGY"})
resp = requests.get(
    WB_API_ENDPOINT.format(wb_codes),
    params={"format": "json", "date": "2010:2020", "per_page": 2000},
    timeout=60,
)

df_gdp = (
    pd.DataFrame(resp.json()[1])
    .rename_column("countryiso3code", "cc3")
    .rename_column("value", "gdppc")
    .select_columns(["cc3", "gdppc"])
    .groupby("cc3", as_index=False)
    .mean()
)
print(f"{len(df_gdp)} countries")
df_gdp.head(3)

57 countries


,cc3,gdppc
0,ALB,4899.168545
1,AND,41694.629450
2,ARG,12024.310542


## QoG - EGDI

In [ ]:
# (
#     pd.read_csv(
#         "https://www.qogdata.pol.gu.se/data/qog_std_ts_jan26.csv",
#         usecols=["ccodealp", "year", "egov_egov"],
#     )
#     .dropna()
#     .to_csv(QOG_CSV, index=False)
# )

In [3]:
# biennial 2011, 2013, 2015, 2017, 2019
df_egdi = (
    pd.read_csv(QOG_CSV)
    .query("2010 <= year <= 2020")
    .rename_column("ccodealp", "cc3")
    .rename_column("egov_egov", "egdi")
    .select_columns(["cc3", "egdi"])
    .groupby("cc3", as_index=False)
    .mean()
)
print(f"{len(df_egdi)} countries")
df_egdi.head(3)

193 countries


,cc3,egdi
0,AFG,0.234058
1,AGO,0.334150
2,ALB,0.589092


## OWID - V-Dem

In [4]:
df_vdem = (
    pd.read_csv(VDEM_URL, storage_options={"User-Agent": "pandas"})
    .clean_names()
    .query("2010 <= year <= 2020")
    .rename_column("code", "cc3")
    .rename_column("electoral_democracy_index", "vdem")
    .select_columns(["cc3", "vdem"])
    .groupby("cc3", as_index=False)
    .mean()
)
print(f"{len(df_vdem)} countries")
df_vdem.head(3)

184 countries


,cc3,vdem
0,AFG,0.341364
1,AGO,0.300000
2,ALB,0.527273


## ITU - GCI

In [ ]:
# import re, time
# ua = {"User-Agent": "Mozilla/5.0", "X-Requested-With": "XMLHttpRequest"}
# rows = []
# for cc3 in df_fes["cc3"]:
#     url = f"https://statbase.org/data/{cc3.lower()}-global-cybersecurity-index/"
#     page = requests.get(url, headers=ua, timeout=15)
#     idind = re.search(r"idind\s*=\s*'?(\d+)", page.text)
#     idc = re.search(r"id_country\s*=\s*'?(\d+)", page.text)
#     if page.status_code != 200 or not (idind and idc):
#         rows.append({"cc3": cc3, "gci_2020": None}); continue
#     tbl = requests.post(
#         "https://statbase.org/include/confcountrynew/get_table_country_en.php",
#         headers={**ua, "Referer": url},
#         data={"indicator": idind.group(1), "country": idc.group(1), "prod": 0, "frmreq": 1},
#         timeout=15,
#     )
#     pairs = dict(re.findall(r'iyt_year">(\d{4})</div><div class="iyt_value">([\d.]+)', tbl.text))
#     rows.append({"cc3": cc3, "gci_2020": float(pairs["2020"]) if "2020" in pairs else None})
#     time.sleep(0.4)
# pd.DataFrame(rows).to_csv(GCI_CSV, index=False)

In [5]:
df_gci = (
    pd.read_csv(GCI_CSV)
    .rename_column("gci_2020", "gci")
)
print(f"{df_gci['gci'].notna().sum()} countries")
df_gci.head(3)

53 countries


,cc3,gci
0,AUS,97.47
1,GBR,99.54
2,JEY,NaN


## CEPII - English official language

In [ ]:
# r = requests.get(
#     "https://github.com/pachadotdev/cepiigeodist/raw/master/data/geo_cepii.rda",
#     timeout=60,
# )
# open("geo_cepii.rda", "wb").write(r.content)
# pyreadr.read_r("geo_cepii.rda")["geo_cepii"].to_csv(CEPII_CSV, index=False)

In [6]:
# ROM is CEPII's pre-2002 code for Romania
df_english = (
    pd.read_csv(CEPII_CSV)
    .rename_column("iso3", "cc3")
    .assign(
        cc3=lambda df_: df_["cc3"].replace({"ROM": "ROU"}),
        english=lambda df_: df_[["langoff_1", "langoff_2", "langoff_3"]]
        .eq("English")
        .any(axis=1)
        .astype(int),
    )
    .select_columns(["cc3", "english"])
    .drop_duplicates("cc3")
)
print(f"{len(df_english)} countries, {df_english['english'].sum()} English-official")
df_english.head(3)

225 countries, 76 English-official


,cc3,english
0,ABW,0
1,AFG,0
2,AGO,0


## Merge and save

In [7]:
df_country = (
    df_fes.merge(df_gdp, on="cc3", how="left", validate="1:1")
    .merge(df_egdi, on="cc3", how="left", validate="1:1")
    .merge(df_vdem, on="cc3", how="left", validate="1:1")
    .merge(df_gci, on="cc3", how="left", validate="1:1")
    .merge(df_english, on="cc3", how="left", validate="1:1")
    .assign(log_gdppc=lambda df_: np.log(df_["gdppc"]))
)
df_country.to_csv(OUT_CSV, index=False)
df_country.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   cc3                59 non-null     object 
 1   country            59 non-null     object 
 2   fe_breach_serious  59 non-null     float64
 3   fe_breach          59 non-null     float64
 4   gdppc              57 non-null     float64
 5   egdi               53 non-null     float64
 6   vdem               53 non-null     float64
 7   gci                53 non-null     float64
 8   english            57 non-null     float64
 9   log_gdppc          57 non-null     float64
dtypes: float64(8), object(2)
memory usage: 4.7+ KB


In [8]:
# countries dropping out
df_country[df_country.isna().any(axis=1)]

,cc3,country,fe_breach_serious,fe_breach,gdppc,egdi,vdem,gci,english,log_gdppc
2,JEY,Jersey,0.788,0.879,NaN,NaN,NaN,NaN,NaN,NaN
3,HKG,Hong-Kong,0.633,0.659,41679.560018,NaN,0.335091,NaN,1.0,10.637766
15,PYF,French-Polynesia,0.330,0.321,21231.435768,NaN,NaN,NaN,0.0,9.963238
26,GGY,Guernsey,0.261,0.274,NaN,NaN,NaN,NaN,NaN,NaN
37,AND,Andorra,0.193,0.182,41694.629450,0.652748,NaN,26.38,0.0,10.638128
38,BMU,Bermuda,0.185,0.206,106178.352397,NaN,NaN,NaN,1.0,11.572876
40,GRL,Greenland,0.181,0.172,49233.872712,NaN,NaN,NaN,0.0,10.804337
